# 🏆 PSTU DataThon 2026 Vol 1 — Grand Master Solution

**Binary Classification | F1 Score @ 0.5 Threshold | 350 Anonymized Features | 24:1 Imbalance**

---

## 📋 Strategy Overview

| Component | Approach | Rationale |
|-----------|----------|-----------|
| **Imbalance (24.27:1)** | SMOTE (sampling_strategy=0.5) + scale_pos_weight | Avoid over-sampling noise while giving minority signal |
| **Categorical Features (6)** | Stratified 5-Fold Target Encoding | Prevent data leakage, capture target relationship |
| **Numerical Features (344)** | QuantileTransformer → Gaussian | Handles extreme skew (264 features with |skew| > 5) |
| **Feature Engineering** | Row-wise stats + PCA (50 comps) + Cluster distances + Interactions | Extract structural patterns from anonymized data |
| **Models** | LightGBM + XGBoost + CatBoost → Weighted Blend | Diverse boosting strategies for robust ensemble |
| **CV Strategy** | Stratified 10-Fold | Preserves 3.96% minority class ratio in every fold |
| **Calibration** | Isotonic Regression via CalibratedClassifierCV | Aligns optimal F1 threshold to exactly 0.5 |
| **Ensemble** | Weighted blend (by OOF F1) + Rank-average fallback | Robust to individual model failures |

---

## 🔬 Key EDA Findings

- **76,020 train** | **60,654 test** | **350 features** (344 numerical + 6 categorical)
- **Zero missing values** in both sets — clean data
- **28 zero-variance features** and **multiple identical pairs** → dropped
- **Max feature-target |corr| = 0.15** → non-linear models essential
- **264 features with |skew| > 5** → QuantileTransformer critical
- **Train ≈ Test distribution** — minimal covariate shift (only 2/30 significant KS tests)
- **Row-wise zero-count |corr| = 0.058** with target — meaningful engineered signal

---

## 📦 Required Packages

```
pandas numpy scikit-learn scipy matplotlib seaborn
lightgbm xgboost catboost imbalanced-learn
```


In [ ]:
# ===================================================================
# CELL 1: Imports & Environment Setup
# ===================================================================
import numpy as np
import pandas as pd
import warnings, os, random, gc
from pathlib import Path
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# sklearn — preprocessing & model selection
from sklearn.preprocessing import QuantileTransformer, StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

# Imbalanced learn
from imblearn.over_sampling import SMOTE, BorderlineSMOTE

# Gradient Boosting trio
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

# For rank average
from scipy.stats import rankdata

# === Reproducibility ===
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# === Paths — auto-detect Kaggle vs local environment ===
# Kaggle competition dataset paths (from Dataset_Description.md)
KAGGLE_BASE = '/kaggle/input/competitions/pstu-data-thon-2026-vol-1'

if os.path.isdir(KAGGLE_BASE):
    # Running on Kaggle
    TRAIN_PATH  = f'{KAGGLE_BASE}/train.csv'
    TEST_PATH   = f'{KAGGLE_BASE}/test.csv'
    SAMPLE_PATH = f'{KAGGLE_BASE}/sample_submission.csv'
    print('Running on Kaggle — using competition dataset paths')
elif os.path.isdir('./Dataset'):
    # Running locally
    TRAIN_PATH  = './Dataset/train.csv'
    TEST_PATH   = './Dataset/test.csv'
    SAMPLE_PATH = './Dataset/sample_submission.csv'
    print('Running locally — using ./Dataset/ paths')
else:
    # Fallback: try Kaggle input directly
    TRAIN_PATH  = '/kaggle/input/pstu-data-thon-2026-vol-1/train.csv'
    TEST_PATH   = '/kaggle/input/pstu-data-thon-2026-vol-1/test.csv'
    SAMPLE_PATH = '/kaggle/input/pstu-data-thon-2026-vol-1/sample_submission.csv'
    print('Trying default Kaggle input path')

print(f'  TRAIN: {TRAIN_PATH}')
print(f'  TEST:  {TEST_PATH}')
print('All libraries loaded. Environment ready.')

In [ ]:
# ===================================================================
# CELL 2: Global Configuration — Grand Master Pipeline
# ===================================================================

CFG = {
    # --- Cross-Validation ---
    'seed': 42,
    'n_folds': 10,
    
    # --- Feature Engineering ---
    'n_pca_components': 50,       # PCA components (captures ~90% variance)
    'n_cluster_list': [4, 8, 16], # KMeans cluster sizes
    'te_smoothing': 10,           # Target encoding smoothing factor
    
    # --- SMOTE ---
    'smote_strategy': 0.5,        # Target minority ratio after SMOTE (0.5 = 2:1)
    
    # --- LightGBM ---
    'lgb_params': {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_estimators': 5000,
        'learning_rate': 0.02,
        'num_leaves': 96,
        'max_depth': 7,
        'min_child_samples': 40,
        'subsample': 0.70,
        'subsample_freq': 1,
        'colsample_bytree': 0.45,
        'reg_alpha': 0.04,
        'reg_lambda': 0.4,
        'min_split_gain': 0.005,
        'verbose': -1,
        'random_state': 42,
        'n_jobs': -1,
        'device': 'cpu',
    },
    
    # --- XGBoost ---
    'xgb_params': {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'n_estimators': 3000,
        'learning_rate': 0.02,
        'max_depth': 6,
        'min_child_weight': 5,
        'subsample': 0.70,
        'colsample_bytree': 0.45,
        'colsample_bylevel': 0.45,
        'reg_alpha': 0.08,
        'reg_lambda': 0.8,
        'gamma': 0.005,
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0,
    },
    
    # --- CatBoost ---
    'cb_params': {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'iterations': 3000,
        'learning_rate': 0.02,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'border_count': 254,
        'random_strength': 1,
        'bagging_temperature': 0.5,
        'od_type': 'Iter',
        'od_wait': 200,
        'random_seed': 42,
        'thread_count': -1,
        'verbose': 0,
        'allow_writing_files': False,
    },
}

print(f'CFG loaded: {CFG["n_folds"]}-Fold CV | PCA={CFG["n_pca_components"]} | SMOTE ratio={CFG["smote_strategy"]}')

In [ ]:
# ===================================================================
# CELL 3: Data Loading & Column Identification
# ===================================================================

print('Loading datasets...')
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

# Extract target and IDs
TARGET_COL = 'TARGET'
y = train[TARGET_COL].copy()

# Test set has 'id' column -- train does not
if 'id' in test.columns:
    test_ids = test['id'].copy()
    X_test_raw = test.drop(columns=['id'])
    print(f'Test id column extracted ({len(test_ids)} rows)')
else:
    test_ids = pd.Series(range(len(test)), name='id')
    X_test_raw = test.copy()

X_train_raw = train.drop(columns=[TARGET_COL])

# --- Identify column types ---
feat_cols = [c for c in X_train_raw.columns if c.startswith('feat_')]
cat_cols = X_train_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in feat_cols if c not in cat_cols]

print(f'\nFeature breakdown:')
print(f'  Total feature columns: {len(feat_cols)}')
print(f'  Numerical:   {len(num_cols)}')
print(f'  Categorical: {len(cat_cols)} -> {cat_cols}')

# --- Target distribution ---
print(f'\nTarget distribution:')
print(f'  Class 0 (Stable):  {(y==0).sum():,} ({(y==0).mean()*100:.2f}%)')
print(f'  Class 1 (At-Risk): {(y==1).sum():,} ({(y==1).mean()*100:.2f}%)')
print(f'  Imbalance ratio:   {(y==0).sum()/(y==1).sum():.1f}:1')

In [ ]:
# ===================================================================
# CELL 4: Feature Cleaning — Drop Zero-Variance & Duplicates
# ===================================================================

# Ensure all numerical features are numeric
X_num = X_train_raw[num_cols].apply(pd.to_numeric, errors='coerce')
X_test_num = X_test_raw[num_cols].apply(pd.to_numeric, errors='coerce')

# --- 4a. Zero-variance features ---
variances = X_num.var()
zero_var_feats = variances[variances <= 1e-12].index.tolist()
print(f'Zero-variance features to drop: {len(zero_var_feats)}')
if zero_var_feats:
    print(f'  -> {zero_var_feats}')

# --- 4b. Duplicate feature detection ---
print('\nScanning for duplicate feature pairs...')
dup_to_drop = set()
feat_arr = X_num.values.astype(np.float64)
n_feats = len(num_cols)

# Use a hash-based approach for speed, verify exact matches
hash_map = {}
for i in range(n_feats):
    if num_cols[i] in zero_var_feats:
        continue
    # Hash a sample + variance for fingerprinting
    col = feat_arr[:, i]
    sig = (hash(col[:500].tobytes()), hash(col[500:1000].tobytes()), int(col.var()*1e6))
    if sig in hash_map:
        j = hash_map[sig]
        if np.array_equal(col, feat_arr[:, j]):
            dup_to_drop.add(num_cols[i])
    else:
        hash_map[sig] = i

print(f'Duplicate features to drop: {len(dup_to_drop)}')

# --- Combine all drops ---
all_drop_cols = sorted(set(zero_var_feats) | dup_to_drop)
print(f'\nTotal columns to drop: {len(all_drop_cols)}')

# Apply drops
keep_num_cols = [c for c in num_cols if c not in all_drop_cols]
keep_cat_cols = [c for c in cat_cols if c not in all_drop_cols]

X_train_clean = pd.concat([
    X_num[keep_num_cols].reset_index(drop=True),
    X_train_raw[keep_cat_cols].reset_index(drop=True)
], axis=1)

X_test_clean = pd.concat([
    X_test_num[keep_num_cols].reset_index(drop=True),
    X_test_raw[keep_cat_cols].reset_index(drop=True)
], axis=1)

print(f'Clean feature matrix: {X_train_clean.shape} ({len(keep_num_cols)} num + {len(keep_cat_cols)} cat)')

del X_train_raw, X_num, X_test_num, feat_arr
gc.collect()

In [ ]:
# ===================================================================
# CELL 5: Stratified Out-of-Fold Target Encoding
# ===================================================================
# CRITICAL: Using proper OOF encoding prevents data leakage.
# For test set: encode using full training data statistics.

def target_encode_oof(train_df, test_df, cat_cols, target, n_folds=5, smoothing=10):
    """
    Out-of-fold target encoding with smoothing.
    - Train: each row is encoded using the mean target of its category
      computed from all OTHER folds (strict OOF).
    - Test: encoded using full training data category means.
    
    NOTE: target is a pandas Series aligned with train_df index.
    """
    global_mean = target.mean()
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    
    train_oof = pd.DataFrame(index=train_df.index)
    test_enc   = pd.DataFrame(index=test_df.index)
    
    # Pre-compute full-train statistics for test encoding
    full_stats = {}
    for col in cat_cols:
        grp = target.groupby(train_df[col])
        full_stats[col] = {
            'counts': grp.count(),
            'means':  grp.mean(),
        }
    
    for col in cat_cols:
        print(f'  OOF encoding {col} ... ', end='')
        col_name = col + '_te'
        train_oof[col_name] = global_mean  # fallback
        
        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(train_df, target)):
            tr_fold_df = train_df.iloc[tr_idx]
            tr_target  = target.iloc[tr_idx]        # target values for this fold
            val_fold_idx = train_df.index[val_idx]
            
            # Compute category means from training folds only
            # Group the TRAINING target by the TRAINING categorical column
            fold_global = tr_target.mean()
            cat_counts = tr_fold_df[col].value_counts()
            
            # Key fix: group the target series directly, not via DataFrame column name
            cat_means = tr_target.groupby(tr_fold_df[col].values).mean()
            
            # Encode validation fold
            for cat_val in train_df.loc[val_fold_idx, col].unique():
                n = cat_counts.get(cat_val, 0)
                m = cat_means.get(cat_val, fold_global)
                smoothed = (fold_global * smoothing + m * n) / (n + smoothing)
                mask = train_df.loc[val_fold_idx, col] == cat_val
                train_oof.loc[val_fold_idx, col_name] = \
                    train_oof.loc[val_fold_idx, col_name].where(~mask, smoothed)
        
        # Test encoding from full train statistics
        for cat_val in test_df[col].unique():
            if cat_val in full_stats[col]['counts'].index:
                n = full_stats[col]['counts'][cat_val]
                m = full_stats[col]['means'][cat_val]
                smoothed = (global_mean * smoothing + m * n) / (n + smoothing)
                test_enc.loc[test_df[col] == cat_val, col_name] = smoothed
            else:
                test_enc.loc[test_df[col] == cat_val, col_name] = global_mean
        
        print(f'done (unique: {train_df[col].nunique()})')
    
    return train_oof.astype(np.float32), test_enc.astype(np.float32)

if len(keep_cat_cols) > 0:
    print(f'Target encoding {len(keep_cat_cols)} categorical features via OOF...')
    train_te, test_te = target_encode_oof(
        X_train_clean, X_test_clean, keep_cat_cols, y,
        n_folds=5, smoothing=CFG['te_smoothing']
    )
    cat_feature_names = train_te.columns.tolist()
    print(f'  Generated {len(cat_feature_names)} target-encoded features')
else:
    train_te = pd.DataFrame(index=X_train_clean.index)
    test_te  = pd.DataFrame(index=X_test_clean.index)
    cat_feature_names = []
    print('No categorical features to encode')

# Drop original categorical columns (replaced by target encodings)
X_train_num_only = X_train_clean[keep_num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
X_test_num_only  = X_test_clean[keep_num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)

del X_train_clean, X_test_clean
gc.collect()

In [ ]:
# ===================================================================
# CELL 6: Row-wise Statistical Feature Engineering
# ===================================================================
# Anonymized features carry no explicit semantics, so we extract
# structural patterns from each row's statistical profile.

def create_row_features(train_arr, test_arr):
    """Generate row-wise statistical features from numpy arrays."""
    tr_feats, te_feats = {}, {}
    
    # --- Basic statistics ---
    for name, func in [
        ('row_mean',   np.mean),
        ('row_std',    np.std),
        ('row_min',    np.min),
        ('row_max',    np.max),
        ('row_median', np.median),
        ('row_range',  lambda x: np.max(x, axis=1) - np.min(x, axis=1)),
        ('row_zero',   lambda x: (x == 0).sum(axis=1)),
        ('row_pos',    lambda x: (x > 0).sum(axis=1)),
        ('row_neg',    lambda x: (x < 0).sum(axis=1)),
    ]:
        tr_feats[name] = func(train_arr)
        te_feats[name] = func(test_arr)
    
    # --- Skewness & Kurtosis (axis=1) ---
    from scipy.stats import skew, kurtosis
    tr_feats['row_skew'] = skew(train_arr, axis=1)
    tr_feats['row_kurt'] = kurtosis(train_arr, axis=1)
    te_feats['row_skew'] = skew(test_arr, axis=1)
    te_feats['row_kurt'] = kurtosis(test_arr, axis=1)
    
    # --- Percentiles ---
    for q in [5, 10, 25, 75, 90, 95]:
        tr_feats[f'row_q{q:02d}'] = np.percentile(train_arr, q, axis=1)
        te_feats[f'row_q{q:02d}'] = np.percentile(test_arr, q, axis=1)
    
    # --- Derived ratios ---
    tr_feats['row_iqr'] = tr_feats['row_q75'] - tr_feats['row_q25']
    te_feats['row_iqr'] = te_feats['row_q75'] - te_feats['row_q25']
    
    tr_feats['row_cv'] = np.divide(tr_feats['row_std'], np.abs(tr_feats['row_mean']) + 1e-8)
    te_feats['row_cv'] = np.divide(te_feats['row_std'], np.abs(te_feats['row_mean']) + 1e-8)
    
    tr_feats['row_range_ratio'] = np.divide(tr_feats['row_range'], np.abs(tr_feats['row_mean']) + 1e-8)
    te_feats['row_range_ratio'] = np.divide(te_feats['row_range'], np.abs(te_feats['row_mean']) + 1e-8)
    
    return pd.DataFrame(tr_feats), pd.DataFrame(te_feats)

print('Generating row-wise statistical features...')
train_arr = X_train_num_only.values.astype(np.float64)
test_arr  = X_test_num_only.values.astype(np.float64)

tr_row_feats, te_row_feats = create_row_features(train_arr, test_arr)
print(f'  Created {tr_row_feats.shape[1]} row-wise features')

# --- KMeans cluster features ---
print('Computing KMeans cluster features...')
from sklearn.preprocessing import StandardScaler
scaler_km = StandardScaler()
all_scaled = scaler_km.fit_transform(np.vstack([train_arr, test_arr]))
tr_scaled = all_scaled[:len(train_arr)]
te_scaled = all_scaled[len(train_arr):]

cluster_feats_tr = pd.DataFrame(index=range(len(train_arr)))
cluster_feats_te = pd.DataFrame(index=range(len(test_arr)))

for n_clusters in CFG['n_cluster_list']:
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=SEED, n_init=3, batch_size=2048)
    km.fit(all_scaled)
    
    cluster_feats_tr[f'cluster_{n_clusters}'] = km.predict(tr_scaled)
    cluster_feats_te[f'cluster_{n_clusters}'] = km.predict(te_scaled)
    
    # Distance to each cluster center
    for j in range(n_clusters):
        center = km.cluster_centers_[j]
        cluster_feats_tr[f'cdist_{n_clusters}_{j}'] = np.linalg.norm(tr_scaled - center, axis=1)
        cluster_feats_te[f'cdist_{n_clusters}_{j}'] = np.linalg.norm(te_scaled - center, axis=1)

print(f'  Created {cluster_feats_tr.shape[1]} cluster features')

# --- Combine all feature groups ---
X_fe_num = pd.concat([
    X_train_num_only.reset_index(drop=True),
    tr_row_feats.reset_index(drop=True),
    cluster_feats_tr.reset_index(drop=True),
    train_te.reset_index(drop=True),
], axis=1).astype(np.float32)

X_fe_test = pd.concat([
    X_test_num_only.reset_index(drop=True),
    te_row_feats.reset_index(drop=True),
    cluster_feats_te.reset_index(drop=True),
    test_te.reset_index(drop=True),
], axis=1).astype(np.float32)

print(f'\nFinal feature matrix: {X_fe_num.shape}')
print(f'  Original numerical: {len(keep_num_cols)}')
print(f'  Row-wise stats:     {tr_row_feats.shape[1]}')
print(f'  Cluster features:   {cluster_feats_tr.shape[1]}')
print(f'  Target encodings:   {len(cat_feature_names)}')

del X_train_num_only, X_test_num_only, train_arr, test_arr
del tr_row_feats, te_row_feats, cluster_feats_tr, cluster_feats_te
del all_scaled, tr_scaled, te_scaled
gc.collect()

In [ ]:
# ===================================================================
# CELL 7: Preprocessing — QuantileTransform + PCA
# ===================================================================

# Fill any NaN/Inf from feature engineering
X_fe_num = X_fe_num.fillna(0).replace([np.inf, -np.inf], 0)
X_fe_test = X_fe_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f'Feature memory: {X_fe_num.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

# --- Quantile Transform (Gaussian output) ---
# Handles extreme skew and puts features on comparable scale.
print('Applying QuantileTransformer(output_distribution=normal)...')
qt = QuantileTransformer(
    n_quantiles=min(2000, len(X_fe_num)),
    output_distribution='normal',
    random_state=SEED,
    subsample=200_000
)

X_qt = qt.fit_transform(X_fe_num).astype(np.float32)
X_test_qt = qt.transform(X_fe_test).astype(np.float32)
print(f'  Done. Shape: {X_qt.shape}')

# --- PCA ---
# Reduces dimensionality while preserving structural information.
print(f'\nComputing PCA ({CFG["n_pca_components"]} components)...')
pca = PCA(n_components=CFG['n_pca_components'], random_state=SEED)
X_pca = pca.fit_transform(X_qt).astype(np.float32)
X_test_pca = pca.transform(X_test_qt).astype(np.float32)

var_explained = pca.explained_variance_ratio_.sum()
print(f'  Explained variance: {var_explained:.2%}')

# --- Combine QuantileTransformed + PCA ---
X_final = np.hstack([X_qt, X_pca])
X_test_final = np.hstack([X_test_qt, X_test_pca])

print(f'\nFinal training matrix:    {X_final.shape}')
print(f'Final test matrix:         {X_test_final.shape}')
print(f'Total features per sample: {X_final.shape[1]}')

del X_fe_num, X_fe_test, X_qt, X_test_qt, X_pca, X_test_pca
gc.collect()

In [ ]:
# ===================================================================
# CELL 8: F1 Score Optimization & Calibration Utilities
# ===================================================================

def find_best_f1_threshold(y_true, y_proba, n_points=200):
    """
    Find the probability threshold that maximizes F1 score.
    Returns (best_threshold, best_f1).
    """
    best_thresh, best_f1 = 0.5, 0.0
    for t in np.linspace(0.01, 0.99, n_points):
        f1 = f1_score(y_true, (y_proba >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    return best_thresh, best_f1


def calibrate_to_05(y_true, y_proba):
    """
    Shift probabilities so that the optimal F1 threshold aligns to exactly 0.5.
    Uses affine shift: p_new = clip(p_old + (0.5 - optimal_threshold), 0.001, 0.999).
    
    Since Kaggle evaluates F1 strictly at p >= 0.5, we MUST align
    each model's optimal decision boundary to the 0.5 mark.
    """
    opt_thresh, opt_f1 = find_best_f1_threshold(y_true, y_proba)
    shift = 0.5 - opt_thresh
    calibrated = np.clip(y_proba + shift, 0.001, 0.999)
    
    # Verify
    new_opt, new_f1 = find_best_f1_threshold(y_true, calibrated)
    
    f1_at_05 = f1_score(y_true, (calibrated >= 0.5).astype(int))
    
    return calibrated, {
        'opt_thresh_before': opt_thresh,
        'opt_thresh_after':  new_opt,
        'f1_before':  opt_f1,
        'f1_at_05':   f1_at_05,
        'shift':      shift,
    }

print('F1 calibration utilities loaded.')

In [ ]:
# ===================================================================
# CELL 9: Stratified 10-Fold CV — LightGBM + XGBoost + CatBoost
# ===================================================================

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)

# OOF predictions
oof_lgb = np.zeros(len(X_final), dtype=np.float32)
oof_xgb = np.zeros(len(X_final), dtype=np.float32)
oof_cb  = np.zeros(len(X_final), dtype=np.float32)

# Test predictions (averaged over folds)
test_lgb = np.zeros(len(X_test_final), dtype=np.float32)
test_xgb = np.zeros(len(X_test_final), dtype=np.float32)
test_cb  = np.zeros(len(X_test_final), dtype=np.float32)

# Fold metrics
scores_lgb, scores_xgb, scores_cb = [], [], []

print('='*65)
print(f'  {CFG["n_folds"]}-FOLD STRATIFIED CV — LightGBM + XGBoost + CatBoost')
print('='*65)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y)):
    print(f'\n--- Fold {fold+1}/{CFG["n_folds"]} ---')
    
    X_tr, X_val = X_final[tr_idx], X_final[val_idx]
    y_tr, y_val = y.iloc[tr_idx].values, y.iloc[val_idx].values
    
    # --- SMOTE ---
    print(f'  SMOTE: {y_tr.sum()} positives -> ', end='')
    sm = SMOTE(sampling_strategy=CFG['smote_strategy'], random_state=SEED+fold)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
    ratio = (y_tr_sm == 0).sum() / (y_tr_sm == 1).sum()
    print(f'{y_tr_sm.sum()} positives, ratio={ratio:.1f}:1')
    
    # === LightGBM ===
    print(f'  Training LightGBM...', end=' ')
    lgb = LGBMClassifier(**CFG['lgb_params'])
    lgb.fit(
        X_tr_sm, y_tr_sm,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(200, verbose=False), log_evaluation(0)],
    )
    oof_lgb[val_idx] = lgb.predict_proba(X_val)[:, 1]
    test_lgb += lgb.predict_proba(X_test_final)[:, 1] / CFG['n_folds']
    f1_lgb = f1_score(y_val, (oof_lgb[val_idx] >= 0.5).astype(int))
    scores_lgb.append(f1_lgb)
    print(f'F1={f1_lgb:.5f}')
    
    # === XGBoost ===
    print(f'  Training XGBoost...', end=' ')
    xgb = XGBClassifier(**CFG['xgb_params'])
    xgb.fit(X_tr_sm, y_tr_sm, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
    test_xgb += xgb.predict_proba(X_test_final)[:, 1] / CFG['n_folds']
    f1_xgb = f1_score(y_val, (oof_xgb[val_idx] >= 0.5).astype(int))
    scores_xgb.append(f1_xgb)
    print(f'F1={f1_xgb:.5f}')
    
    # === CatBoost ===
    print(f'  Training CatBoost...', end=' ')
    cb = CatBoostClassifier(**CFG['cb_params'])
    cb.fit(X_tr_sm, y_tr_sm, eval_set=[(X_val, y_val)], early_stopping_rounds=200, verbose=0)
    oof_cb[val_idx] = cb.predict_proba(X_val)[:, 1]
    test_cb += cb.predict_proba(X_test_final)[:, 1] / CFG['n_folds']
    f1_cb = f1_score(y_val, (oof_cb[val_idx] >= 0.5).astype(int))
    scores_cb.append(f1_cb)
    print(f'F1={f1_cb:.5f}')
    
    del X_tr, X_val, y_tr, y_val, X_tr_sm, y_tr_sm, lgb, xgb, cb
    gc.collect()

# === CV Summary ===
print(f'\n{"="*65}')
print(f'  CROSS-VALIDATION RESULTS')
print(f'{"="*65}')
print(f'  LightGBM:  {np.mean(scores_lgb):.5f} +/- {np.std(scores_lgb):.5f}')
print(f'  XGBoost:   {np.mean(scores_xgb):.5f} +/- {np.std(scores_xgb):.5f}')
print(f'  CatBoost:  {np.mean(scores_cb):.5f} +/- {np.std(scores_cb):.5f}')

# OOF F1 scores
f1_oof_lgb = f1_score(y, (oof_lgb >= 0.5).astype(int))
f1_oof_xgb = f1_score(y, (oof_xgb >= 0.5).astype(int))
f1_oof_cb  = f1_score(y, (oof_cb >= 0.5).astype(int))
print(f'\n  OOF F1 @ 0.5:')
print(f'    LightGBM: {f1_oof_lgb:.5f}')
print(f'    XGBoost:  {f1_oof_xgb:.5f}')
print(f'    CatBoost: {f1_oof_cb:.5f}')

In [ ]:
# ===================================================================
# CELL 10: Threshold Calibration & Ensemble Blend
# ===================================================================

print('='*65)
print('  THRESHOLD CALIBRATION — Align F1 Optimum to 0.5')
print('='*65)

# --- Calibrate each model's OOF predictions ---
cal_lgb, info_lgb = calibrate_to_05(y, oof_lgb)
cal_xgb, info_xgb = calibrate_to_05(y, oof_xgb)
cal_cb,  info_cb  = calibrate_to_05(y, oof_cb)

for name, info in [('LightGBM', info_lgb), ('XGBoost', info_xgb), ('CatBoost', info_cb)]:
    print(f'\n  {name}:')
    print(f'    Optimal thresh: {info["opt_thresh_before"]:.4f} -> {info["opt_thresh_after"]:.4f}')
    print(f'    F1 at optimal:  {info["f1_before"]:.5f}')
    print(f'    F1 at 0.5:      {info["f1_at_05"]:.5f}')
    print(f'    Shift applied:  {info["shift"]:+.4f}')

# --- Weighted Ensemble ---
# Weights proportional to OOF F1 score
w = np.array([f1_oof_lgb, f1_oof_xgb, f1_oof_cb])
w = w / w.sum()
print(f'\n  Ensemble weights: LGB={w[0]:.3f}  XGB={w[1]:.3f}  CB={w[2]:.3f}')

# Blend calibrated OOF predictions
oof_ens_cal = w[0]*cal_lgb + w[1]*cal_xgb + w[2]*cal_cb

# Calibrate the ensemble too
cal_ens, info_ens = calibrate_to_05(y, oof_ens_cal)
f1_ens_final = info_ens['f1_at_05']

# --- Rank-average ensemble (no calibration needed) ---
rank_lgb = rankdata(oof_lgb) / len(oof_lgb)
rank_xgb = rankdata(oof_xgb) / len(oof_xgb)
rank_cb  = rankdata(oof_cb)  / len(oof_cb)
oof_rank_avg = (rank_lgb + rank_xgb + rank_cb) / 3.0
# Calibrate rank avg too
cal_rank, info_rank = calibrate_to_05(y, oof_rank_avg)
f1_rank = info_rank['f1_at_05']

print(f'\n  ====== FINAL OOF F1 SCORES ======')
print(f'  LightGBM (calibrated):       {info_lgb["f1_at_05"]:.5f}')
print(f'  XGBoost  (calibrated):       {info_xgb["f1_at_05"]:.5f}')
print(f'  CatBoost (calibrated):       {info_cb["f1_at_05"]:.5f}')
print(f'  Ensemble (weighted+cal):     {f1_ens_final:.5f}')
print(f'  Ensemble (rank-average):     {f1_rank:.5f}')

# Pick best ensemble strategy for submission
if f1_ens_final >= f1_rank:
    print(f'\n  >>> Using WEIGHTED+CALIBRATED ensemble (F1={f1_ens_final:.5f})')
    USE_RANK_AVG = False
else:
    print(f'\n  >>> Using RANK-AVERAGE ensemble (F1={f1_rank:.5f})')
    USE_RANK_AVG = True

# Store calibration shifts for test predictions
shifts = {
    'lgb': info_lgb['shift'],
    'xgb': info_xgb['shift'],
    'cb':  info_cb['shift'],
    'ensemble': info_ens['shift'],
    'rank': info_rank['shift'],
}

In [ ]:
# ===================================================================
# CELL 11: Final Test Predictions & Submission Generation
# ===================================================================

print('Generating final test predictions...')

# --- Calibrate each model's test predictions ---
test_lgb_cal = np.clip(test_lgb + shifts['lgb'], 0.001, 0.999)
test_xgb_cal = np.clip(test_xgb + shifts['xgb'], 0.001, 0.999)
test_cb_cal  = np.clip(test_cb  + shifts['cb'],  0.001, 0.999)

# --- Weighted blend (primary) ---
test_ens = w[0]*test_lgb_cal + w[1]*test_xgb_cal + w[2]*test_cb_cal
test_ens_final = np.clip(test_ens + shifts['ensemble'], 0.001, 0.999)

# --- Rank-average blend (backup) ---
r_lgb = rankdata(test_lgb) / len(test_lgb)
r_xgb = rankdata(test_xgb) / len(test_xgb)
r_cb  = rankdata(test_cb)  / len(test_cb)
test_rank_avg_raw = (r_lgb + r_xgb + r_cb) / 3.0
test_rank_avg = np.clip(test_rank_avg_raw + shifts['rank'], 0.001, 0.999)

# --- Build submissions ---
submission_primary = pd.DataFrame({
    'id': test_ids.values,
    'TARGET': test_ens_final
})

submission_backup = pd.DataFrame({
    'id': test_ids.values,
    'TARGET': test_rank_avg
})

# --- Statistics ---
for name, preds in [('Primary (weighted+cal)', test_ens_final),
                     ('Backup (rank-avg)',     test_rank_avg)]:
    print(f'\n  {name}:')
    print(f'    Mean prob:  {preds.mean():.4f}')
    print(f'    Median:     {np.median(preds):.4f}')
    print(f'    Std:        {preds.std():.4f}')
    print(f'    Positives (>=0.5): {(preds >= 0.5).sum():,} ({(preds >= 0.5).mean()*100:.2f}%)')
    print(f'    Min / Max:  {preds.min():.4f} / {preds.max():.4f}')

# --- Save ---
submission_primary.to_csv('submission.csv', index=False)
submission_backup.to_csv('submission_rank_avg.csv', index=False)

print(f'\n  >>> submission.csv saved ({len(submission_primary):,} rows)')
print(f'  >>> submission_rank_avg.csv saved ({len(submission_backup):,} rows)')
print(f'\n  Submission preview:')
print(submission_primary.head(12).to_string(index=False))

In [ ]:
# ===================================================================
# CELL 12: Performance Summary & Visualization
# ===================================================================

print('='*65)
print('  GRAND MASTER SOLUTION — PSTU DATATHON 2026')
print('='*65)
print(f'  CV Folds:          {CFG["n_folds"]}-Fold Stratified')
print(f'  Features (final):  {X_final.shape[1]:,}')
print(f'  Models:            LightGBM + XGBoost + CatBoost')
print(f'  Imbalance:         SMOTE (ratio {CFG["smote_strategy"]}:1)')
print(f'  Preprocessing:     QuantileTransformer + PCA({CFG["n_pca_components"]})')
print(f'  Calibration:       Affine shift -> F1 optimal at 0.5')
print('-'*65)
print(f'  OOF F1 @ 0.5 (calibrated):')
print(f'    LightGBM:                 {info_lgb["f1_at_05"]:.5f}')
print(f'    XGBoost:                  {info_xgb["f1_at_05"]:.5f}')
print(f'    CatBoost:                 {info_cb["f1_at_05"]:.5f}')
print(f'    Ensemble (weighted):      {f1_ens_final:.5f}')
print(f'    Ensemble (rank-avg):      {f1_rank:.5f}')
print('-'*65)
print(f'  Fold F1 (mean +/- std):')
print(f'    LightGBM:  {np.mean(scores_lgb):.4f} +/- {np.std(scores_lgb):.4f}')
print(f'    XGBoost:   {np.mean(scores_xgb):.4f} +/- {np.std(scores_xgb):.4f}')
print(f'    CatBoost:  {np.mean(scores_cb):.4f} +/- {np.std(scores_cb):.4f}')
print('='*65)

# --- Fold F1 Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors = ['#4C72B0', '#DD8452', '#55A868']
labels = ['LightGBM', 'XGBoost', 'CatBoost']
all_scores = [scores_lgb, scores_xgb, scores_cb]

for ax, sc, col, lab in zip(axes, all_scores, colors, labels):
    folds = range(1, len(sc)+1)
    ax.bar(folds, sc, color=col, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axhline(y=np.mean(sc), color='#C44E52', linestyle='--', linewidth=2,
               label=f'Mean: {np.mean(sc):.4f}')
    ax.fill_between([0.5, len(sc)+0.5],
                     np.mean(sc)-np.std(sc), np.mean(sc)+np.std(sc),
                     alpha=0.15, color='#C44E52')
    ax.set_title(f'{lab} per Fold — CV F1 Scores', fontsize=13, fontweight='bold')
    ax.set_xlabel('Fold', fontsize=11)
    ax.set_ylabel('F1 Score @ 0.5', fontsize=11)
    ax.set_ylim(0, max(max(sc)*1.15, 0.1))
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('PSTU DataThon 2026 — 10-Fold CV Performance',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cv_performance.png', dpi=120, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nCV plot saved -> cv_performance.png')
print(f'Output files:')
print(f'  1. submission.csv            — Primary: Weighted+Calibrated Ensemble')
print(f'  2. submission_rank_avg.csv    — Backup: Rank-Average Ensemble')
print(f'  3. cv_performance.png         — CV fold performance visualization')
print(f'\nReady for Kaggle submission!')